# T45-曲线 - 债券收益率曲线计算系统

## 项目概述

本系统用于构建债券收益率曲线，通过 **Hermite 插值法** 计算任意期限的收益率曲线。

### 主要功能
1. 基于中债曲线数据进行 Hermite 插值
2. 支持城投债、产业债、二永债等多种债券类型
3. 按发行方式、是否永续、信用评级等维度分类计算
4. 计算信用债相对于基准曲线的利差

### 数据源
- `bond.marketinfo_curve`: 曲线市场数据（中债估值）
- `bond.basicinfo_curve`: 曲线基础信息（分类、期限）
- `bond.marketinfo_credit`: 信用债市场数据
- `bond.basicinfo_credit`: 信用债基础信息

---
## 1. 环境配置

导入所需依赖库和配置参数。

In [ ]:
# -*- coding: utf-8 -*-
"""
导入依赖库
"""
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import datetime
from datetime import datetime, timedelta

# 数据库相关
import pymysql
from sqlalchemy import create_engine, text
from sqlalchemy import pool as sqlalchemy_pool

# 导入配置
from config import (
    get_mysql_connection_string,
    STANDARD_TERMS,
    TARGET_TERMS,
    VALID_RATINGS,
    INTERPOLATION_PRECISION,
    TERM_DIFFERENCE_TOLERANCE
)

print("依赖导入完成")

In [ ]:
# 创建数据库连接引擎
connection_string = get_mysql_connection_string()
sql_engine = create_engine(
    connection_string,
    poolclass=sqlalchemy_pool.NullPool,
    pool_recycle=3600
)

print(f"数据库连接已创建: {sql_engine.url.host}/{sql_engine.url.database}")

In [ ]:
# 初始化标准期限点参数
TERMS = np.array(STANDARD_TERMS)
MIN_TERM = min(TERMS)
MAX_TERM = max(TERMS)
TERM_LENGTH = len(TERMS)

print(f"标准期限点数量: {TERM_LENGTH}")
print(f"期限范围: {MIN_TERM} - {MAX_TERM} 年")

---
## 2. 数据获取

从数据库读取曲线数据和信用债数据。

In [ ]:
def get_curve_data(dt):
    """
    获取中债曲线数据
    
    Parameters:
    -----------
    dt : str
        日期，格式：'YYYY-MM-DD'
    
    Returns:
    --------
    pd.DataFrame
        曲线数据，包含：日期、收益率、发行方式、是否二永、隐含评级、曲线期限、大类
    """
    sql = f"""
    -- 城投债曲线
    SELECT
        A.dt AS 日期,
        A.收益率,
        B.发行方式,
        B.是否二永,
        B.隐含评级,
        B.曲线期限,
        '城投' AS 大类
    FROM (
        SELECT DT AS dt, trade_code, CLOSE AS 收益率
        FROM bond.marketinfo_curve
        WHERE dt = '{dt}'
        AND TRADE_CODE IN (
            SELECT trade_code FROM bond.basicinfo_curve 
            WHERE classify2 LIKE '%中债%' AND classify2 LIKE '%城投债%'
        )
    ) A
    LEFT JOIN (
        SELECT 
            TRADE_CODE,
            CASE WHEN classify2 LIKE '%非公开%' THEN '私募' ELSE '公募' END AS 发行方式,
            CASE WHEN classify2 LIKE '%可续期%' THEN '是' ELSE '否' END AS 是否二永,
            SUBSTRING(
                REPLACE(classify2, '＋', '+'),
                LOCATE('(', REPLACE(classify2, '＋', '+')) + 1,
                CHAR_LENGTH(classify2) - LOCATE('(', REPLACE(classify2, '＋', '+')) - 1
            ) AS 隐含评级,
            LEFT(SUBSTRING_INDEX(SEC_NAME, ':', -1), 
                CHAR_LENGTH(SUBSTRING_INDEX(SEC_NAME, ':', -1)) - 1
            ) * (
                (RIGHT(SEC_NAME, 1) = '天') / 365 +
                (RIGHT(SEC_NAME, 1) = '月') / 12 +
                (RIGHT(SEC_NAME, 1) = '年')
            ) AS 曲线期限
        FROM bond.basicinfo_curve
    ) B ON A.trade_code = B.TRADE_CODE
    
    UNION
    
    -- 产业债曲线
    SELECT
        A.dt AS 日期,
        A.收益率,
        B.发行方式,
        B.是否二永,
        B.隐含评级,
        B.曲线期限,
        '产业' AS 大类
    FROM (
        SELECT dt, trade_code, CLOSE AS 收益率
        FROM bond.marketinfo_curve
        WHERE dt = '{dt}'
        AND TRADE_CODE IN (
            SELECT trade_code FROM bond.basicinfo_curve 
            WHERE classify2 LIKE '%中债%' AND (classify2 LIKE '%产业%' OR classify2 LIKE '%企业%')
        )
    ) A
    LEFT JOIN (
        SELECT 
            TRADE_CODE,
            CASE WHEN classify2 LIKE '%非公开%' THEN '私募' ELSE '公募' END AS 发行方式,
            CASE WHEN classify2 LIKE '%次级可续期%' THEN '是' ELSE '否' END AS 是否二永,
            SUBSTRING(
                REPLACE(classify2, '＋', '+'),
                LOCATE('(', REPLACE(classify2, '＋', '+')) + 1,
                CHAR_LENGTH(classify2) - LOCATE('(', REPLACE(classify2, '＋', '+')) - 1
            ) AS 隐含评级,
            LEFT(SUBSTRING_INDEX(SEC_NAME, ':', -1), 
                CHAR_LENGTH(SUBSTRING_INDEX(SEC_NAME, ':', -1)) - 1
            ) * (
                (RIGHT(SEC_NAME, 1) = '天') / 365 +
                (RIGHT(SEC_NAME, 1) = '月') / 12 +
                (RIGHT(SEC_NAME, 1) = '年')
            ) AS 曲线期限
        FROM bond.basicinfo_curve
    ) B ON A.trade_code = B.TRADE_CODE
    """
    
    with sql_engine.begin() as conn:
        df = pd.read_sql(sql, conn)
    
    return df


def get_credit_bond_data(dt, ratings=None):
    """
    获取信用债市场数据
    
    Parameters:
    -----------
    dt : str
        日期，格式：'YYYY-MM-DD'
    ratings : list, optional
        隐含评级列表，默认使用 VALID_RATINGS
    
    Returns:
    --------
    pd.DataFrame
        信用债数据
    """
    if ratings is None:
        ratings = VALID_RATINGS
    
    ratings_str = "','".join(ratings)
    
    sql = f"""
    SELECT
        dt AS 日期,
        trade_code,
        ths_bond_balance_bond AS 债券余额,
        ths_evaluate_yield_cb_bond_exercise AS 中债行权估值,
        CASE 
            WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0 
            THEN ths_evaluate_modified_dur_cb_bond_exercise
            ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_spread_durcb_bond_exercise
        END AS 中债行权剩余期限,
        ths_cb_market_implicit_rating_bond AS 隐含评级
    FROM bond.marketinfo_credit
    WHERE dt = '{dt}'
        AND ths_evaluate_yield_cb_bond_exercise > 0
        AND ths_cb_market_implicit_rating_bond IN ('{ratings_str}')
        AND (ths_evaluate_modified_dur_cb_bond_exercise > 0 
             OR ths_evaluate_interest_durcb_bond_exercise > 0)
    """
    
    with sql_engine.begin() as conn:
        df = pd.read_sql(sql, conn)
    
    return df


def get_target_term_data():
    """
    获取目标期限配置
    
    Returns:
    --------
    pd.DataFrame
        目标期限数据
    """
    sql = "SELECT term as 目标期限 FROM yq.目标期限1"
    
    with sql_engine.begin() as conn:
        df = pd.read_sql(sql, conn)
    
    return df


print("数据获取函数定义完成")

---
## 3. 数据处理

数据清洗和分类信息提取。

In [ ]:
def clean_curve_data(df):
    """
    清洗曲线数据
    
    Parameters:
    -----------
    df : pd.DataFrame
        原始曲线数据
    
    Returns:
    --------
    pd.DataFrame
        清洗后的曲线数据
    """
    # 复制数据避免修改原数据
    df = df.copy()
    
    # 过滤无效收益率
    df = df[df['收益率'] > 0]
    
    # 过滤无效期限
    df = df[df['曲线期限'].notna() & (df['曲线期限'] > 0)]
    
    # 日期格式转换
    df['日期'] = pd.to_datetime(df['日期'])
    
    return df


def clean_credit_bond_data(df):
    """
    清洗信用债数据
    
    Parameters:
    -----------
    df : pd.DataFrame
        原始信用债数据
    
    Returns:
    --------
    pd.DataFrame
        清洗后的信用债数据
    """
    df = df.copy()
    
    # 过滤无效估值
    df = df[df['中债行权估值'] > 0]
    
    # 过滤无效期限
    df = df[df['中债行权剩余期限'].notna() & (df['中债行权剩余期限'] > 0)]
    
    # 日期格式转换
    df['日期'] = pd.to_datetime(df['日期'])
    
    # 债券余额转换为数值
    df['债券余额'] = pd.to_numeric(df['债券余额'], errors='coerce')
    
    return df


def correct_city_invest_flag(df):
    """
    修正是否城投标识
    
    规则:
    - AA(2) 且标记为非城投 -> 改为城投
    - AAA- 且标记为城投 -> 改为非城投
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含'隐含评级'和'是否城投'列的数据
    
    Returns:
    --------
    pd.DataFrame
        修正后的数据
    """
    df = df.copy()
    
    # AA(2) 评级通常是城投债
    mask_aa2 = (df['隐含评级'] == 'AA(2)') & (df['是否城投'] == '否')
    df.loc[mask_aa2, '是否城投'] = '是'
    
    # AAA- 通常是产业债
    mask_aaa = (df['隐含评级'] == 'AAA-') & (df['是否城投'] == '是')
    df.loc[mask_aaa, '是否城投'] = '否'
    
    return df


print("数据处理函数定义完成")

---
## 4. 核心逻辑

Hermite 插值函数和曲线计算函数。

In [ ]:
def hermite_interpolation(x, x1, x2, y1, y2, d1, d2, precision=INTERPOLATION_PRECISION):
    """
    三次 Hermite 插值
    
    使用三次 Hermite 插值计算目标期限的收益率。
    
    Parameters:
    -----------
    x : float or array-like
        目标期限（待插值点）
    x1 : float or array-like
        左侧锚点期限
    x2 : float or array-like
        右侧锚点期限
    y1 : float or array-like
        左侧锚点收益率
    y2 : float or array-like
        右侧锚点收益率
    d1 : float or array-like
        左侧锚点导数（斜率）
    d2 : float or array-like
        右侧锚点导数（斜率）
    precision : int, optional
        结果保留小数位数，默认4位
    
    Returns:
    --------
    float or array-like
        插值后的收益率
    
    Notes:
    ------
    Hermite 基函数:
    - H1(t) = 1 - 3t^2 + 2t^3  (对应 y1)
    - H2(t) = 3t^2 - 2t^3       (对应 y2)
    - H3(t) = t - 2t^2 + t^3    (对应 d1)
    - H4(t) = -t^2 + t^3        (对应 d2)
    
    插值公式:
    y(x) = y1*H1(t) + y2*H2(t) + d1*H3(t) + d2*H4(t)
    其中 t = (x - x1) / (x2 - x1)
    """
    h = x2 - x1  # 区间长度
    t = (x - x1) / h  # 归一化参数
    
    # 计算 Hermite 基函数
    H1 = 1 - 3 * t**2 + 2 * t**3
    H2 = 3 * t**2 - 2 * t**3
    H3 = t - 2 * t**2 + t**3
    H4 = -t**2 + t**3
    
    # 插值公式
    y = y1 * H1 + y2 * H2 + d1 * H3 + d2 * H4
    
    return np.round(y, precision)


print("Hermite插值函数定义完成")

In [ ]:
def find_anchor_points(term_values, terms=TERMS):
    """
    查找插值锚点
    
    对于每个目标期限，找到4个锚点:
    - x1: 小于等于目标期限的最大标准期限
    - x2: 大于等于目标期限的最小标准期限
    - x11: x1 的前一个标准期限
    - x22: x2 的后一个标准期限
    
    Parameters:
    -----------
    term_values : array-like
        目标期限值数组
    terms : array-like, optional
        标准期限点数组
    
    Returns:
    --------
    dict
        包含 x1, x2, x11, x22 的字典
    """
    min_term = min(terms)
    max_term = max(terms)
    term_length = len(terms)
    
    # 判断是否在标准期限点上或超出范围
    mask_in_term = np.logical_or(
        np.in1d(term_values, terms),
        np.logical_or(term_values < min_term, term_values > max_term)
    )
    
    # 对于区间外的点，查找锚点
    out_term_mask = ~mask_in_term
    
    # x1: 小于等于目标期限的最大标准期限
    x1 = np.max(
        np.where(term_values[:, np.newaxis] >= terms, terms, min_term),
        axis=1
    )
    
    # x2: 大于等于目标期限的最小标准期限
    x2 = np.min(
        np.where(term_values[:, np.newaxis] <= terms, terms, max_term),
        axis=1
    )
    
    # 使用 searchsorted 查找索引位置
    insert_points_x1 = np.searchsorted(terms, x1, side='left')
    insert_points_x2 = np.searchsorted(terms, x2, side='right')
    
    # x11: x1 的前一个标准期限
    x11 = terms[np.where(insert_points_x1 > 0, insert_points_x1 - 1, 0)]
    
    # x22: x2 的后一个标准期限
    x22 = terms[np.where(insert_points_x2 < term_length, insert_points_x2, term_length - 1)]
    
    return {
        'x1': x1,
        'x2': x2,
        'x11': x11,
        'x22': x22,
        'mask_in_term': mask_in_term,
        'out_term_mask': out_term_mask
    }


print("锚点查找函数定义完成")

In [ ]:
def calculate_curve(data_curve, data_target):
    """
    计算曲线收益率
    
    使用 Hermite 插值法计算任意期限的收益率曲线。
    
    Parameters:
    -----------
    data_curve : pd.DataFrame
        原始曲线数据（包含标准期限点的收益率）
    data_target : pd.DataFrame
        目标曲线数据（需要计算收益率的期限点）
    
    Returns:
    --------
    pd.DataFrame
        包含计算好的曲线收益率的数据
    """
    # 复制数据
    data_target = data_target.copy()
    
    # 获取目标期限值
    curve_term_values = data_target['曲线期限'].values
    
    # 区分区间内和区间外的点
    mask_in_term = np.logical_or(
        np.in1d(curve_term_values, TERMS),
        np.logical_or(curve_term_values < MIN_TERM, curve_term_values > MAX_TERM)
    )
    
    index_in_term = data_target[mask_in_term].index
    index_out_term = data_target[~mask_in_term].index
    
    # ========== 处理区间内点（直接匹配） ==========
    def merge_yield(index, term_col, yield_col):
        """合并收益率数据"""
        merge_target = data_target.loc[index].join(
            data_curve.set_index(['日期', '隐含评级', '大类', '发行方式', '是否二永', '曲线期限']),
            on=['日期', '隐含评级', '大类', '发行方式', '是否二永', term_col],
            how='left',
            rsuffix='_1'
        )
        valid_idx = merge_target['收益率'] > 0
        if not merge_target[valid_idx].empty:
            data_target.loc[merge_target.index[valid_idx], yield_col] = merge_target.loc[valid_idx, '收益率']
    
    # 区间内点直接赋值
    merge_yield(index_in_term, '曲线期限', '曲线收益率')
    
    # ========== 处理区间外点（需要插值） ==========
    if len(index_out_term) > 0:
        out_term_values = curve_term_values[~mask_in_term]
        
        # 查找锚点
        anchors = find_anchor_points(out_term_values)
        
        # 存储锚点期限
        data_target.loc[index_out_term, 'x1'] = anchors['x1']
        data_target.loc[index_out_term, 'x2'] = anchors['x2']
        data_target.loc[index_out_term, 'x11'] = anchors['x11']
        data_target.loc[index_out_term, 'x22'] = anchors['x22']
        
        # 获取锚点收益率
        merge_yield(index_out_term, 'x1', 'yx1')
        merge_yield(index_out_term, 'x2', 'yx2')
        merge_yield(index_out_term, 'x11', 'yx11')
        merge_yield(index_out_term, 'x22', 'yx22')
        
        # 计算导数（使用中心差分）
        data_target.loc[index_out_term, 'd1'] = (
            data_target.loc[index_out_term, 'yx2'] - data_target.loc[index_out_term, 'yx11']
        ) / (2 * (data_target.loc[index_out_term, 'x2'] - data_target.loc[index_out_term, 'x11']))
        
        data_target.loc[index_out_term, 'd2'] = (
            data_target.loc[index_out_term, 'yx22'] - data_target.loc[index_out_term, 'yx1']
        ) / (2 * (data_target.loc[index_out_term, 'x22'] - data_target.loc[index_out_term, 'x1']))
        
        # 处理边界情况（x22 缺失）
        df_out = data_target.loc[index_out_term]
        null_mask = df_out['yx22'].isnull() | df_out['d2'].isnull()
        
        if null_mask.any():
            null_index = df_out[null_mask].index
            data_target.loc[null_index, 'yx22'] = data_target.loc[null_index, 'yx2']
            data_target.loc[null_index, 'x22'] = data_target.loc[null_index, 'x2']
            # 重新计算 d2
            data_target.loc[null_index, 'd2'] = (
                data_target.loc[null_index, 'yx2'] - data_target.loc[null_index, 'yx1']
            ) / (2 * (data_target.loc[null_index, 'x2'] - data_target.loc[null_index, 'x1']))
        
        # 执行 Hermite 插值
        data_target.loc[index_out_term, '曲线收益率'] = hermite_interpolation(
            data_target.loc[index_out_term, '曲线期限'],
            data_target.loc[index_out_term, 'x1'],
            data_target.loc[index_out_term, 'x2'],
            data_target.loc[index_out_term, 'yx1'],
            data_target.loc[index_out_term, 'yx2'],
            data_target.loc[index_out_term, 'd1'],
            data_target.loc[index_out_term, 'd2']
        )
    
    return data_target


print("曲线计算函数定义完成")

In [ ]:
def calculate_standardized_yield(data_main, data_curve):
    """
    计算标准化收益率
    
    将债券收益率调整到目标期限:
    目标期限收益率 = 中债行权估值 + (目标期限曲线收益率 - 原期限曲线收益率)
    
    Parameters:
    -----------
    data_main : pd.DataFrame
        信用债数据
    data_curve : pd.DataFrame
        曲线数据
    
    Returns:
    --------
    pd.DataFrame
        包含标准化收益率的数据
    """
    data_main = data_main.copy()
    
    # 获取债券期限值
    bond_term_values = data_main['中债行权剩余期限'].values
    
    # 区分区间内和区间外
    mask_in_term = np.logical_or(
        np.in1d(bond_term_values, TERMS),
        np.logical_or(bond_term_values < MIN_TERM, bond_term_values > MAX_TERM)
    )
    
    # 匹配曲线数据的函数
    def merge_curve_data(index, term_col, yield_col, curve_data=data_curve):
        """合并曲线收益率"""
        df = data_main.loc[index]
        merged = df.join(
            curve_data.set_index(['日期', '隐含评级', '大类', '发行方式', '是否二永', '曲线期限']),
            on=['日期', '隐含评级', '是否城投', '发行方式', '是否二永', term_col],
            how='left',
            rsuffix='_1'
        )
        valid_idx = merged['收益率'] > 0
        if not merged[valid_idx].empty:
            data_main.loc[merged.index[valid_idx], yield_col] = merged.loc[valid_idx, '收益率']
    
    # ========== 计算目标期限曲线收益率 ==========
    # 对所有债券，匹配目标期限的曲线收益率
    merge_curve_data(data_main.index, '目标期限', '目标期限曲线收益率')
    
    # ========== 计算原期限曲线收益率 ==========
    index_in_term = data_main[mask_in_term].index
    index_out_term = data_main[~mask_in_term].index
    
    # 区间内点直接匹配
    merge_curve_data(index_in_term, '中债行权剩余期限', '原期限曲线收益率')
    
    # 区间外点需要插值
    if len(index_out_term) > 0:
        out_term_values = bond_term_values[~mask_in_term]
        
        # 查找锚点
        anchors = find_anchor_points(out_term_values)
        
        data_main.loc[index_out_term, 'x1'] = anchors['x1']
        data_main.loc[index_out_term, 'x2'] = anchors['x2']
        data_main.loc[index_out_term, 'x11'] = anchors['x11']
        data_main.loc[index_out_term, 'x22'] = anchors['x22']
        
        # 获取锚点收益率
        merge_curve_data(index_out_term, 'x1', 'yx1')
        merge_curve_data(index_out_term, 'x2', 'yx2')
        merge_curve_data(index_out_term, 'x11', 'yx11')
        merge_curve_data(index_out_term, 'x22', 'yx22')
        
        # 计算导数
        data_main.loc[index_out_term, 'd1'] = (
            data_main.loc[index_out_term, 'yx2'] - data_main.loc[index_out_term, 'yx11']
        ) / (2 * (data_main.loc[index_out_term, 'x2'] - data_main.loc[index_out_term, 'x11']))
        
        data_main.loc[index_out_term, 'd2'] = (
            data_main.loc[index_out_term, 'yx22'] - data_main.loc[index_out_term, 'yx1']
        ) / (2 * (data_main.loc[index_out_term, 'x22'] - data_main.loc[index_out_term, 'x1']))
        
        # 执行插值
        data_main.loc[index_out_term, '原期限曲线收益率'] = hermite_interpolation(
            data_main.loc[index_out_term, '中债行权剩余期限'],
            data_main.loc[index_out_term, 'x1'],
            data_main.loc[index_out_term, 'x2'],
            data_main.loc[index_out_term, 'yx1'],
            data_main.loc[index_out_term, 'yx2'],
            data_main.loc[index_out_term, 'd1'],
            data_main.loc[index_out_term, 'd2']
        )
    
    # ========== 计算标准化收益率 ==========
    data_main['标准化收益率'] = (
        data_main['中债行权估值'] + 
        (data_main['目标期限曲线收益率'] - data_main['原期限曲线收益率'])
    )
    
    return data_main


print("标准化收益率计算函数定义完成")

---
## 5. 执行与测试

执行主流程并验证结果。

In [ ]:
# 设置测试日期
TEST_DATE = '2024-08-22'

print(f"测试日期: {TEST_DATE}")

In [ ]:
# 获取曲线数据
print("获取曲线数据...")
curve_data = get_curve_data(TEST_DATE)
curve_data = clean_curve_data(curve_data)

print(f"曲线数据量: {len(curve_data)}")
print(f"曲线数据列: {list(curve_data.columns)}")
curve_data.head()

In [ ]:
# 获取目标期限配置
print("获取目标期限配置...")
target_terms = get_target_term_data()

print(f"目标期限数量: {len(target_terms)}")
print(f"目标期限范围: {target_terms['目标期限'].min()} - {target_terms['目标期限'].max()}")

In [ ]:
# 测试 Hermite 插值函数
print("测试 Hermite 插值...")

# 示例：计算1.5年期限的收益率
# 假设：1年收益率3.5%，2年收益率3.8%
x = 1.5  # 目标期限
x1, x2 = 1, 2  # 锚点
y1, y2 = 3.5, 3.8  # 锚点收益率

# 假设导数
d1 = 0.3
d2 = 0.4

result = hermite_interpolation(x, x1, x2, y1, y2, d1, d2)
print(f"1.5年期收益率插值结果: {result}%")

In [ ]:
# 测试锚点查找
print("测试锚点查找...")

test_terms = np.array([0.1, 0.5, 1.5, 3.7, 25])
anchors = find_anchor_points(test_terms)

print("测试期限:", test_terms)
print("x1（左锚点）:", anchors['x1'])
print("x2（右锚点）:", anchors['x2'])
print("x11（左前锚点）:", anchors['x11'])
print("x22（右后锚点）:", anchors['x22'])

In [ ]:
# 获取信用债数据（可选，需要数据库连接）
try:
    print("获取信用债数据...")
    credit_data = get_credit_bond_data(TEST_DATE)
    credit_data = clean_credit_bond_data(credit_data)
    
    print(f"信用债数据量: {len(credit_data)}")
    print(f"信用债数据列: {list(credit_data.columns)}")
    
    # 显示数据样例
    credit_data.head()
except Exception as e:
    print(f"获取信用债数据时出错: {e}")
    print("跳过信用债数据处理步骤")

In [ ]:
# 计算加权平均收益率示例
def calculate_weighted_yield(df, group_cols=['日期', '目标期限']):
    """
    计算按债券余额加权的平均收益率
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含'债券余额'和'标准化收益率'的数据
    group_cols : list
        分组列
    
    Returns:
    --------
    pd.DataFrame
        加权平均收益率
    """
    weighted = df.groupby(group_cols).apply(
        lambda x: (x['债券余额'] * x['标准化收益率']).sum() / x['债券余额'].sum()
    ).reset_index(name='加权收益率')
    
    return weighted


print("加权收益率计算函数定义完成")

---
## 6. 工具函数

可复用的辅助函数。

In [ ]:
def get_date_range(start_date, end_date):
    """
    获取日期范围内的交易日列表
    
    Parameters:
    -----------
    start_date : str
        开始日期，格式：'YYYY-MM-DD'
    end_date : str
        结束日期，格式：'YYYY-MM-DD'
    
    Returns:
    --------
    list
        日期列表
    """
    sql = f"""
    SELECT DISTINCT dt 
    FROM bond.marketinfo_curve
    WHERE dt BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY dt
    """
    
    with sql_engine.begin() as conn:
        df = pd.read_sql(sql, conn)
    
    return df['dt'].tolist()


def filter_by_term_difference(df, tolerance=TERM_DIFFERENCE_TOLERANCE):
    """
    过滤期限差异过大的数据
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含'目标期限'和'中债行权剩余期限'的数据
    tolerance : float
        期限差异容忍度（年）
    
    Returns:
    --------
    pd.DataFrame
        过滤后的数据
    """
    return df[abs(df['目标期限'] - df['中债行权剩余期限']) < tolerance]


def adjust_10year_node(df):
    """
    调整10年节点
    
    如果存在10.01和10年两个期限点，使用差值调整所有大于10年的收益率。
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含'目标期限'和'加权收益率'的数据
    
    Returns:
    --------
    pd.DataFrame
        调整后的数据
    """
    df = df.copy()
    
    def adjust_group(group):
        if 10.01 in group['目标期限'].values and 10 in group['目标期限'].values:
            yield_10_01 = group[group['目标期限'] == 10.01]['加权收益率'].iloc[0]
            yield_10 = group[group['目标期限'] == 10]['加权收益率'].iloc[0]
            
            difference = yield_10_01 - yield_10
            
            if not pd.isna(difference):
                group.loc[group['目标期限'] > 10, '加权收益率'] -= difference
        
        return group
    
    return df.groupby('日期').apply(adjust_group).reset_index(drop=True)


print("工具函数定义完成")

In [ ]:
# 验证函数列表
print("="*50)
print("函数列表验证")
print("="*50)

functions = [
    'hermite_interpolation',
    'find_anchor_points', 
    'calculate_curve',
    'calculate_standardized_yield',
    'get_curve_data',
    'get_credit_bond_data',
    'get_target_term_data',
    'clean_curve_data',
    'clean_credit_bond_data',
    'correct_city_invest_flag',
    'calculate_weighted_yield',
    'get_date_range',
    'filter_by_term_difference',
    'adjust_10year_node'
]

for func_name in functions:
    if func_name in dir():
        print(f"[OK] {func_name}")
    else:
        print(f"[MISSING] {func_name}")

print("\n" + "="*50)
print("配置验证")
print("="*50)
print(f"标准期限点数量: {len(STANDARD_TERMS)}")
print(f"期限范围: {min(STANDARD_TERMS)} - {max(STANDARD_TERMS)} 年")
print(f"有效评级: {VALID_RATINGS}")
print(f"插值精度: {INTERPOLATION_PRECISION} 位小数")
print(f"期限差异容忍度: {TERM_DIFFERENCE_TOLERANCE} 年")

---
## 总结

### 核心功能
1. **Hermite 插值**：使用三次 Hermite 插值计算任意期限收益率
2. **多维度分类**：支持城投/产业、公募/私募、永续/非永续等分类
3. **标准化收益率**：将债券收益率调整到标准期限
4. **加权平均**：按债券余额计算加权平均收益率

### 关键公式
- **Hermite 插值**: `y(x) = y1*H1(t) + y2*H2(t) + d1*H3(t) + d2*H4(t)`
- **标准化收益率**: `目标期限收益率 = 估值 + (目标期限曲线 - 原期限曲线)`
- **导数计算**: `d1 = (y2 - y11) / (2 * (x2 - x11))`

### 注意事项
1. 所有敏感信息通过环境变量配置
2. 边界情况需要特殊处理（如30年附近）
3. 期限差异超过容忍度的数据会被过滤